## Smoke Test

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

In [2]:
# Locate project root directory (containing src/ and configs/)
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv()

True

In [3]:
# Check if all 3 API keys loaded (masked prefixes only)
for env_name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
    val = os.getenv(env_name)
    status = f"OK ({val[:6]}...)" if val else "MISSING!"
    print(f"{env_name:<20}: {status}")

OPENAI_API_KEY      : OK (sk-pro...)
ANTHROPIC_API_KEY   : OK (sk-ant...)
GEMINI_API_KEY      : OK (AQ.Ab8...)


### Execution: N=2

__Stateless batch collection via run_batch() (3 models x 2 tasks x 2 runs = 12 samples)__

Persist raw outputs to data/raw/smoke_n2/

In [4]:
from src.api_execution.run_batch import run_batch

run_batch(
    n_iterations=2,
    batch_name="smoke_n2",
    prompts_cfg_path="configs/prompts_v1.0.yaml",
    models_cfg_path="configs/models_baseline.yaml",
    inter_call_delay_sec=1.0,
)


=== Model: GPT-6 Luna (gpt-6-luna) -> data\raw\smoke_n2\gpt6_luna.jsonl ===
  [RUN] T1_gpt6_luna_001 ... OK (11.80s, 430 words)
  [RUN] T1_gpt6_luna_002 ... OK (6.68s, 413 words)
  [RUN] T2_gpt6_luna_001 ... OK (10.29s, 449 words)
  [RUN] T2_gpt6_luna_002 ... OK (9.33s, 499 words)

=== Model: Claude Haiku 4.5 (claude-haiku-4-5) -> data\raw\smoke_n2\claude_haiku45.jsonl ===
  [RUN] T1_claude_haiku45_001 ... OK (8.47s, 461 words)
  [RUN] T1_claude_haiku45_002 ... OK (7.65s, 404 words)
  [RUN] T2_claude_haiku45_001 ... OK (8.03s, 429 words)
  [RUN] T2_claude_haiku45_002 ... OK (8.45s, 471 words)

=== Model: Gemini 3.5 Flash Lite (gemini-3.5-flash-lite) -> data\raw\smoke_n2\gemini35_flash_lite.jsonl ===
  [RUN] T1_gemini35_flash_lite_001 ... OK (3.35s, 418 words)
  [RUN] T1_gemini35_flash_lite_002 ... OK (2.80s, 435 words)
  [RUN] T2_gemini35_flash_lite_001 ... OK (2.96s, 409 words)
  [RUN] T2_gemini35_flash_lite_002 ... OK (2.86s, 439 words)


__Validate persisted JSONL quality and word count distribution__

Check success status, 400–450 word range, and preview a sample response

In [5]:
import pandas as pd
from pathlib import Path
from src.schema import load_records_jsonl

raw_files = sorted(Path("data/raw/smoke_n2").glob("*.jsonl"))
all_records = []
for f in raw_files:
    all_records.extend(load_records_jsonl(f))

df = pd.DataFrame([r.to_dict() for r in all_records])
display(df[["sample_id", "task_id", "model_id", "status", "latency_sec", "word_count"]])

# Check 1st sample response
if not df.empty:
    print("\n--- Sample Output Preview ---")
    print(df.iloc[0]["raw_output"][:300] + "...")

,sample_id,task_id,model_id,status,latency_sec,word_count
0,T1_claude_haiku45_001,Task1_KNN,claude-haiku-4-5,success,8.470,461
1,T1_claude_haiku45_002,Task1_KNN,claude-haiku-4-5,success,7.652,404
2,T2_claude_haiku45_001,Task2_AIEthics,claude-haiku-4-5,success,8.025,429
3,T2_claude_haiku45_002,Task2_AIEthics,claude-haiku-4-5,success,8.454,471
4,T1_gemini35_flash_lite_001,Task1_KNN,gemini-3.5-flash-lite,success,3.347,418
5,T1_gemini35_flash_lite_002,Task1_KNN,gemini-3.5-flash-lite,success,2.804,435
6,T2_gemini35_flash_lite_001,Task2_AIEthics,gemini-3.5-flash-lite,success,2.965,409
7,T2_gemini35_flash_lite_002,Task2_AIEthics,gemini-3.5-flash-lite,success,2.863,439
8,T1_gpt6_luna_001,Task1_KNN,gpt-6-luna,success,11.800,430
9,T1_gpt6_luna_002,Task1_KNN,gpt-6-luna,success,6.677,413



--- Sample Output Preview ---
# K-Nearest Neighbors (KNN): A Simple Yet Powerful Algorithm

## What is KNN?

K-Nearest Neighbors is one of the simplest and most intuitive machine learning algorithms. The core idea is straightforward: **classify a new data point based on the class of its nearest neighbors**. It's a non-parametric...


__Test one-stop export of master and blind CSVs__

Confirm jsonl_to_csv properly outputs shuffled blind records and private key map

In [6]:
from src.preprocessing.export_annotations import jsonl_to_csv

exported = jsonl_to_csv(
    input_paths="data/raw/smoke_n2",
    output_dir="data/annotations/smoke_n2",
    batch_prefix="smoke_n2",
    seed=42,
)
print("Exported files:", exported)

Exported files: {'master_csv': WindowsPath('data/annotations/smoke_n2/smoke_n2_master.csv'), 'blind_csv': WindowsPath('data/annotations/smoke_n2/smoke_n2_blind.csv'), 'blind_key_csv': WindowsPath('data/annotations/smoke_n2/smoke_n2_blind_blind_key.csv')}


In [7]:
# Preview blind table (data/annotations/smoke_n2/smoke_n2_blind.csv)
# Verify model_id and sample_id are stripped, leaving only BLIND_XXX
blind_df = pd.read_csv(exported["blind_csv"])
display(blind_df[["blind_id", "task_id", "word_count"]].head(6))

,blind_id,task_id,word_count
0,BLIND_001,Task2_AIEthics,439
1,BLIND_002,Task1_KNN,435
2,BLIND_003,Task2_AIEthics,429
3,BLIND_004,Task1_KNN,430
4,BLIND_005,Task1_KNN,413
5,BLIND_006,Task2_AIEthics,409


### Execution: N=15

In [8]:
records = [r.to_dict() for f in Path('data/raw/pilot_n15').glob('*.jsonl') for r in load_records_jsonl(f)]
df = pd.DataFrame(records)
print(df.groupby(['model_id', 'task_id', 'status']).size())
print('\nWord count summary:\n', df.groupby(['model_id', 'task_id'])['word_count'].agg(['count', 'mean', 'std', 'min', 'max']).round(1))


model_id               task_id         status 
claude-haiku-4-5       Task1_KNN       success    15
                       Task2_AIEthics  success    15
gemini-3.5-flash-lite  Task1_KNN       success    15
                       Task2_AIEthics  success    15
gpt-6-luna             Task1_KNN       success    15
                       Task2_AIEthics  success    15
dtype: int64

Word count summary:
                                       count   mean   std  min  max
model_id              task_id                                     
claude-haiku-4-5      Task1_KNN          15  437.0  14.9  415  462
                      Task2_AIEthics     15  441.6  14.6  417  467
gemini-3.5-flash-lite Task1_KNN          15  437.1  20.8  417  499
                      Task2_AIEthics     15  424.1  28.8  373  485
gpt-6-luna            Task1_KNN          15  413.1  20.1  380  445
                      Task2_AIEthics     15  448.8  20.9  402  476
